In [1]:
!pip install -qU pip awscli boto3 'sagemaker<3.0' transformers==4.9.1
!pip install nvidia-pyindex
!pip install tritonclient[http]

  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [62 lines of output]
      /tmp/pip-build-env-43lje9cn/overlay/lib/python3.11/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running bdist_wheel
      running build
      running build_py
      creating build/lib.linux-x86_64-cpython-311/

In [2]:
import boto3, json, sagemaker, time
from sagemaker import get_execution_role

sess = boto3.Session()
sm = sess.client("sagemaker")
sagemaker_session = sagemaker.Session(boto_session=sess)
role = get_execution_role()
client = boto3.client("sagemaker-runtime")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [4]:
region = 'ap-south-1'
base = "amazonaws.com.cn" if region.startswith("cn-") else "amazonaws.com"
triton_image_uri = "763104351884.dkr.ecr.{region}.{base}/sagemaker-tritonserver:24.03-py3".format(
    region=region, base=base
)

In [5]:
def create_model(sm_model_name, model_uri, root_folder_name, triton_image_uri=triton_image_uri, role=role):
    container = {
        "Image": triton_image_uri,
        "ModelDataUrl": model_uri,
        "Environment": {"SAGEMAKER_TRITON_DEFAULT_MODEL_NAME": root_folder_name},
    }
    
    create_model_response = sm.create_model(
        ModelName=sm_model_name, ExecutionRoleArn=role, PrimaryContainer=container
    )

    print("Model Arn: " + create_model_response["ModelArn"])

In [6]:
def create_endpoint_config(endpoint_config_name, sm_model_name, instance_type="ml.g4dn.4xlarge"):
    create_endpoint_config_response = sm.create_endpoint_config(
        EndpointConfigName=endpoint_config_name,
        ProductionVariants=[
            {
                "InstanceType": instance_type,
                "InitialVariantWeight": 1,
                "InitialInstanceCount": 1,
                "ModelName": sm_model_name,
                "VariantName": "AllTraffic",
            }
        ],
    )
    
    print("Endpoint Config Arn: " + create_endpoint_config_response["EndpointConfigArn"])

In [7]:
def create_endpoint(endpoint_name, endpoint_config_name):
    create_endpoint_response = sm.create_endpoint(
        EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name
    )
    
    print("Endpoint Arn: " + create_endpoint_response["EndpointArn"])

In [ ]:
def poll(endpoint_name):
    resp = sm.describe_endpoint(EndpointName=endpoint_name)
    status = resp["EndpointStatus"]
    print("Status: " + status)
    
    while status == "Creating":
        time.sleep(60)
        resp = sm.describe_endpoint(EndpointName=endpoint_name)
        status = resp["EndpointStatus"]
        print("Status: " + status)
    
    print("Arn: " + resp["EndpointArn"])
    print("Status: " + status)

In [ ]:
def cleanup(sm_model_name, endpoint_config_name, endpoint_name):
    sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    sm.delete_model(ModelName=sm_model_name)
    sm.delete_endpoint(EndpointName=endpoint_name)